# Practice: data with SQL
*AI from scratch, stage 4* · lesson: https://ihor.sh/ai/data-with-sql.html

Practice extracting and shaping real machine learning training tables with SQL:
filtering populations with `WHERE`, aggregating historical summaries with `GROUP BY`,
joining relational metadata safely with `LEFT JOIN`, and computing point-in-time cumulative
features with window functions to prevent future data leakage.
Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)`
are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Run pd.read_sql_query("SELECT COUNT(*) AS total FROM outages", conn)', 'total_rows = int(pd.read_sql_query("SELECT COUNT(*) AS total FROM outages", conn)["total"].iloc[0])'], 2: ["Run SELECT * FROM outages WHERE kind = 'EL' AND finished = 1", 'df_el = pd.read_sql_query("SELECT * FROM outages WHERE kind = \'EL\' AND finished = 1", conn)'], 3: ['SELECT station, COUNT(*) AS outage_count FROM outages GROUP BY station ORDER BY outage_count DESC LIMIT 5'], 4: ['Use LEFT JOIN stations s ON o.station = s.name'], 5: ['Use ROW_NUMBER() OVER (PARTITION BY equipment ORDER BY start) AS seq'], 6: ['Use COUNT(*) OVER (PARTITION BY station ORDER BY start ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prior_station_outages']}
_ANSWERS = {1: 'dG90YWxfcm93cyA9IGludChwZC5yZWFkX3NxbF9xdWVyeSgiU0VMRUNUIENPVU5UKCopIEFTIHRvdGFsIEZST00gb3V0YWdlcyIsIGNvbm4pWyJ0b3RhbCJdLmlsb2NbMF0p', 2: 'ZGZfZWwgPSBwZC5yZWFkX3NxbF9xdWVyeSgiU0VMRUNUICogRlJPTSBvdXRhZ2VzIFdIRVJFIGtpbmQgPSAnRUwnIEFORCBmaW5pc2hlZCA9IDEiLCBjb25uKQ==', 3: 'ZGZfdG9wX3N0YXRpb25zID0gcGQucmVhZF9zcWxfcXVlcnkoIiIiCiAgICBTRUxFQ1Qgc3RhdGlvbiwgQ09VTlQoKikgQVMgb3V0YWdlX2NvdW50CiAgICBGUk9NIG91dGFnZXMKICAgIEdST1VQIEJZIHN0YXRpb24KICAgIE9SREVSIEJZIG91dGFnZV9jb3VudCBERVNDCiAgICBMSU1JVCA1CiIiIiwgY29ubik=', 4: 'ZGZfam9pbmVkID0gcGQucmVhZF9zcWxfcXVlcnkoIiIiCiAgICBTRUxFQ1Qgby5lcXVpcG1lbnQsIG8uc3RhdGlvbiwgcy5yYWlscm9hZAogICAgRlJPTSBvdXRhZ2VzIG8KICAgIExFRlQgSk9JTiBzdGF0aW9ucyBzIE9OIG8uc3RhdGlvbiA9IHMubmFtZQoiIiIsIGNvbm4p', 5: 'ZGZfc2VxID0gcGQucmVhZF9zcWxfcXVlcnkoIiIiCiAgICBTRUxFQ1QgZXF1aXBtZW50LCBzdGFydCwKICAgICAgICAgICBST1dfTlVNQkVSKCkgT1ZFUiAoUEFSVElUSU9OIEJZIGVxdWlwbWVudCBPUkRFUiBCWSBzdGFydCkgQVMgc2VxCiAgICBGUk9NIG91dGFnZXMKIiIiLCBjb25uKQ==', 6: 'ZGZfcGl0ID0gcGQucmVhZF9zcWxfcXVlcnkoIiIiCiAgICBTRUxFQ1QgZXF1aXBtZW50LCBzdGF0aW9uLCBzdGFydCwKICAgICAgICAgICBDT1VOVCgqKSBPVkVSICgKICAgICAgICAgICAgICAgUEFSVElUSU9OIEJZIHN0YXRpb24KICAgICAgICAgICAgICAgT1JERVIgQlkgc3RhcnQKICAgICAgICAgICAgICAgUk9XUyBCRVRXRUVOIFVOQk9VTkRFRCBQUkVDRURJTkcgQU5EIDEgUFJFQ0VESU5HCiAgICAgICAgICAgKSBBUyBwcmlvcl9zdGF0aW9uX291dGFnZXMKICAgIEZST00gb3V0YWdlcwoiIiIsIGNvbm4p'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import json, sqlite3, urllib.request
import pandas as pd

conn = sqlite3.connect(":memory:")

# Load the real NYC subway outage dataset into SQLite
outages_url = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/outages.csv"
df_outages = pd.read_csv(outages_url)
df_outages.to_sql("outages", conn, index=False, if_exists="replace")

# Load rail station metadata into SQLite
stations_url = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/rail-stations.json"
req = urllib.request.urlopen(stations_url)
st_raw = json.loads(req.read().decode())
pd.DataFrame(st_raw["rows"], columns=st_raw["columns"]).to_sql("stations", conn, index=False, if_exists="replace")

## 1. Basic querying: counting records
Query the total number of records in the `outages` table using `SELECT COUNT(*) AS total FROM outages`.
Store the integer result in `total_rows`.

In [ ]:
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...

In [ ]:
check(1, lambda: total_rows == 667, "outages table should contain 667 rows")

## 2. Filtering populations: WHERE
Query all elevator (`kind = 'EL'`) outages that are completed (`finished = 1`).
Select all columns and assign the resulting DataFrame to `df_el`.

In [ ]:
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...

In [ ]:
check(2, lambda: df_el is not None and len(df_el) > 0 and (df_el["kind"] == "EL").all() and (df_el["finished"].isin([1, True])).all(), "WHERE kind = 'EL' AND finished = 1 keeps only completed elevator outages")

## 3. Aggregation: GROUP BY and HAVING
Find the top 5 stations with the most total outages.
Return two columns: `station` and `outage_count` (using `COUNT(*) AS outage_count`),
ordered by `outage_count DESC LIMIT 5`. Assign to `df_top_stations`.

In [ ]:
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...

In [ ]:
check(3, lambda: df_top_stations is not None and len(df_top_stations) == 5 and "station" in df_top_stations.columns and "outage_count" in df_top_stations.columns and df_top_stations["outage_count"].is_monotonic_decreasing, "GROUP BY station with COUNT(*) and ORDER BY DESC returns the top 5 stations")

## 4. Joining metadata: LEFT JOIN
Enrich the outages table with railroad operator metadata from the `stations` table.
Join `outages o` with `stations s` on `o.station = s.name`.
Select `o.equipment`, `o.station`, and `s.railroad`. Assign the DataFrame to `df_joined`.

In [ ]:
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...

In [ ]:
check(4, lambda: df_joined is not None and len(df_joined) == 667 and "railroad" in df_joined.columns, "LEFT JOIN keeps all 667 outage records while adding railroad metadata where available")

## 5. Window functions: chronological sequence numbers
For each outage, compute which occurrence it is for that specific piece of equipment over time.
Use `ROW_NUMBER() OVER (PARTITION BY equipment ORDER BY start) AS seq`.
Select `equipment`, `start`, and `seq`. Assign to `df_seq`.

In [ ]:
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...

In [ ]:
check(5, lambda: df_seq is not None and "seq" in df_seq.columns and df_seq["seq"].min() == 1 and df_seq["seq"].max() > 1, "ROW_NUMBER() partitions outages by equipment and numbers each incident chronologically")

## 6. Point-in-time feature extraction without leakage
To predict whether an outage will take long to fix, an engineer wants to know how many outages
occurred at that station *prior* to this one.
Use:
`COUNT(*) OVER (PARTITION BY station ORDER BY start ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prior_station_outages`
Select `equipment`, `station`, `start`, and `prior_station_outages`. Assign to `df_pit`.

In [ ]:
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...

In [ ]:
check(6, lambda: df_pit is not None and "prior_station_outages" in df_pit.columns and df_pit["prior_station_outages"].fillna(0).min() == 0, "Point-in-time window counts historical incidents strictly before each row's start time without leakage")